# Kaggle CLI Training

Notebook flow:
1. clone the training repo from GitHub,
2. download `gilkeyio/AudioMNIST` from Hugging Face,
3. save it to `/kaggle/working/data/AudioMNIST`,
4. run training via CLI with Hydra overrides.

Before running:
- enable Internet in Kaggle,
- make sure the Kaggle image has enough disk space for the dataset cache and saved HF dataset.


In [ ]:
import importlib.util
import subprocess
import sys

required = {
    "hydra": "hydra-core",
    "omegaconf": "omegaconf",
    "datasets": "datasets",
    "huggingface_hub": "huggingface_hub",
    "sklearn": "scikit-learn",
}

missing = [package for module, package in required.items() if importlib.util.find_spec(module) is None]

if missing:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *missing])

print("Installed:" if missing else "Already available:", missing)


In [ ]:
from pathlib import Path
import shutil

REPO_URL = "https://github.com/aibryanov/speech-emo-recognition.git"
REPO_REF = "master"
HF_DATASET_NAME = "gilkeyio/AudioMNIST"

REPO_DIR = Path("/kaggle/working/speech-emo-recognition")
DATA_DIR = Path("/kaggle/working/data")
DATASET_DIR = DATA_DIR / "AudioMNIST"

CLI_OVERRIDES = [
    f"dataset.paths.local_path={DATASET_DIR.as_posix()}",
    "train.epochs=5",
    "dataloader.batch_size=64",
    "train.log_every=50",
]

DATA_DIR.mkdir(parents=True, exist_ok=True)


def run(cmd, cwd=None):
    print("$", " ".join(cmd))
    subprocess.run(cmd, cwd=cwd, check=True)


In [ ]:
if REPO_DIR.exists():
    shutil.rmtree(REPO_DIR)

run(["git", "clone", "--depth", "1", "--branch", REPO_REF, REPO_URL, str(REPO_DIR)])
print(f"Repo cloned to: {REPO_DIR}")


In [ ]:
from datasets import load_dataset, load_from_disk

if DATASET_DIR.exists():
    try:
        dataset = load_from_disk(str(DATASET_DIR))
        print(f"Dataset already exists at: {DATASET_DIR}")
    except Exception:
        shutil.rmtree(DATASET_DIR)
        dataset = load_dataset(HF_DATASET_NAME)
        dataset.save_to_disk(str(DATASET_DIR))
        print(f"Dataset re-downloaded to: {DATASET_DIR}")
else:
    dataset = load_dataset(HF_DATASET_NAME)
    dataset.save_to_disk(str(DATASET_DIR))
    print(f"Dataset downloaded to: {DATASET_DIR}")

print(dataset)


In [ ]:
cmd = [sys.executable, "main.py", *CLI_OVERRIDES]
run(cmd, cwd=REPO_DIR)


In [ ]:
outputs_dir = REPO_DIR / "outputs"
if outputs_dir.exists():
    print("Hydra outputs:")
    for path in sorted(outputs_dir.rglob("*"))[-20:]:
        print(path)
else:
    print("Hydra outputs directory was not found.")
